# データベースの基礎 第4回（参考資料）：SQL を試す（SQLite）
* Docker の PostgreSQL（主環境）が使えないときの**補助環境**です
* SQLite という軽量の DBMS を使います（Google Colab でそのまま動きます）
* 演習で使う SQL は PostgreSQL でも SQLite でもまったく同じです．違うのは起動のしかただけです
* サンプルデータは2組あります．`sample`（日本語の5表）はこのノートブックに組み込んであり，`Chinook`（音楽販売の11表）は学生ポータルから自動でダウンロードします．どちらも PostgreSQL 版と同じ表名・列名です

## Google Colab の使い方（初めての人へ）

**開き方**

1. Google アカウントにログインした状態で，学生ポータルの第4回ページにあるリンクを開く（このノートブックが Google Colab で開く）
2. 「警告：このノートブックは Google が作成したものではありません」と出たら「このまま実行」を選ぶ
3. 自分用に保存したいときは，メニュー「ファイル → ドライブにコピーを保存」を選ぶ（保存しなくても実行はできる）

**基本操作**

| 操作 | 方法 |
|------|------|
| セルの実行 | セルをクリックして `Shift + Enter`，またはセル左の再生ボタン |
| すべて実行 | メニュー「ランタイム → すべてのセルを実行」 |
| セルの追加 | 上部メニューの「+ コード」または「+ テキスト」 |

**注意**

* セルは**上から順に**実行する．最初の「環境作成」と「サンプルデータの作成」のセルを飛ばすと，あとのセルがエラーになる
* しばらく操作しないとランタイム（実行環境）が切断される．切断されたら「ランタイム → すべてのセルを実行」でやり直せばよい（メモリ上のデータベースなので，作り直しが必要）

## 環境作成
* Google Colab で使うときは，最初に以下のセルで jupysql をインストールしてください
* jupysql はセルに SQL を入れて実行できる仕組みです
* SQLite はインストールなしで使えます

In [ ]:
# jupysqlをインストール
%pip install jupysql

### JupySQL拡張をロード

In [ ]:
%load_ext sql

### SQLite（メモリDB）に接続

In [ ]:
# SQLite（メモリDB）に接続
%sql sqlite:///:memory:

### 動作確認
* 主環境（PostgreSQL）では `SELECT version();` で確認しますが，SQLite では関数名が違います
* `3.xx.x` のようなバージョン番号が表示されれば動いています

In [ ]:
%%sql
SELECT sqlite_version();

## サンプルデータの作成
* 配布している `sample.sql` と同じ SQL です（表を作り直してから，データを入れる）
* 何度実行してもよい（データは初期状態に戻る）
* 5つの表ができる：部門・従業員・部品・業者・供給

In [ ]:
%%sql
DROP TABLE IF EXISTS 供給;
DROP TABLE IF EXISTS 従業員;
DROP TABLE IF EXISTS 業者;
DROP TABLE IF EXISTS 部品;
DROP TABLE IF EXISTS 部門;

CREATE TABLE 部門 (
    部門番号  INTEGER PRIMARY KEY,
    部門名    TEXT NOT NULL);
CREATE TABLE 従業員 (
    従業員番号  INTEGER PRIMARY KEY,
    部門番号  INTEGER NOT NULL,
    氏名  TEXT NOT NULL,
    住所  TEXT,
    職級  INTEGER);
CREATE TABLE 部品 (
    部品番号  INTEGER PRIMARY KEY,
    部品名    TEXT NOT NULL);
CREATE TABLE 業者 (
    業者番号  INTEGER PRIMARY KEY,
    業者名  TEXT NOT NULL,
    住所  TEXT,
    電話番号  TEXT);
CREATE TABLE 供給 (
    部門番号  INTEGER NOT NULL,
    部品番号  INTEGER NOT NULL,
    業者番号  INTEGER NOT NULL,
    単価  INTEGER NOT NULL,
    数量    INTEGER NOT NULL);

INSERT INTO 部門 VALUES (1, '営業部');
INSERT INTO 部門 VALUES (2, '開発部');

INSERT INTO 従業員 VALUES (1, 1, '佐藤一郎', '横浜市', 2);
INSERT INTO 従業員 VALUES (2, 1, '田中次郎', '静岡市', 1);
INSERT INTO 従業員 VALUES (3, 2, '鈴木三郎', '横浜市', 2);
INSERT INTO 従業員 VALUES (4, 2, '山田四郎', '東京都', 1);

INSERT INTO 部品 VALUES (1, 'CPU');
INSERT INTO 部品 VALUES (2, 'HDD');
INSERT INTO 部品 VALUES (3, 'SSD');
INSERT INTO 部品 VALUES (4, 'ビデオカード');

INSERT INTO 業者 VALUES (1, 'インテル', 'アメリカ合衆国', '000-000-0001');
INSERT INTO 業者 VALUES (2, 'AMD', 'アメリカ合衆国', '000-000-0002');
INSERT INTO 業者 VALUES (3, 'Western Digital', 'アメリカ合衆国', '000-000-0003');

INSERT INTO 供給 VALUES (2, 1, 1, 10000, 100);
INSERT INTO 供給 VALUES (2, 1, 2,  9000,  50);
INSERT INTO 供給 VALUES (2, 2, 3,  5000, 150);
INSERT INTO 供給 VALUES (2, 3, 1,  6000,  60);
INSERT INTO 供給 VALUES (2, 3, 3,  5000,  90);
INSERT INTO 供給 VALUES (2, 4, 2, 15000, 150);


### できた表を確認

In [ ]:
%%sql
SELECT * FROM 従業員;

In [ ]:
%%sql
SELECT * FROM 供給;

## サンプルデータの追加：Chinook（音楽販売）
* 音楽販売ビジネスを模したサンプルデータベース（artist・album・track・customer・invoice など11表，曲は約3500件）
* 学生ポータルで配布している `chinook.sqlite3` を自動でダウンロードし，さきほどのデータベースに**取り付け**（ATTACH）ます
* 取り付けたあとは，sample の表と同じように表名だけで使えます（PostgreSQL の `chinook_pg.sql` を入れた状態と同じ）

In [ ]:
# chinook.sqlite3 を学生ポータル（GitHub）からダウンロード（すでにあれば何もしない）
import os, urllib.request
URL = "https://raw.githubusercontent.com/ggszk/ggszk-lab-public/main/database_basics/DB_04/chinook.sqlite3"
if not os.path.exists("chinook.sqlite3"):
    urllib.request.urlretrieve(URL, "chinook.sqlite3")
print("size:", os.path.getsize("chinook.sqlite3"), "bytes")

In [ ]:
%%sql
ATTACH DATABASE 'chinook.sqlite3' AS chinook;

### できた表を確認

In [ ]:
%%sql
SELECT first_name, last_name, title FROM employee;

In [ ]:
%%sql
-- 曲の数
SELECT COUNT(*) FROM track;

## SQL を試す
* 以下は例です．セルの SQL を書き換えて，いろいろ試してください
* 複数の SQL 文を1つのセルに書いた場合，表示されるのは最後の文の結果だけです

In [ ]:
%%sql
-- 従業員の氏名だけを取り出す（射影）
SELECT 氏名 FROM 従業員;

In [ ]:
%%sql
-- 部品番号が 1 の部品の名前（選択＋射影）
SELECT 部品名 FROM 部品 WHERE 部品番号 = 1;

In [ ]:
%%sql
-- 数量が 70 以上の供給（選択）
SELECT * FROM 供給 WHERE 数量 >= 70;

In [ ]:
%%sql
-- 業者の名前と住所（射影）
SELECT 業者名, 住所 FROM 業者;

In [ ]:
%%sql
-- Chinook：アルバム名とアーティスト名（結合）．最初の10件
SELECT album.title, artist.name
FROM album JOIN artist ON album.artist_id = artist.artist_id
LIMIT 10;

### 自分で書いてみる
* 下のセルを書き換えて実行する．セルを増やしたいときは「+ コード」で追加し，先頭行に `%%sql` と書く

In [ ]:
%%sql
-- ここに自分の SQL を書く（下の1行は消してよい）
SELECT * FROM 部門;